In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
import config
import era5_api
import era5_utils

In [2]:
# era5_api.check_cds_connection()

In [3]:
ds = era5_api.download_era5(
    variables=["2m_temperature"],
    area=config.BBOX_CHIANG_MAI_CDS,
    year=2025, month=3, day=15,
)
ds

[cache] Reading era5land_2m_temperature_20_98_17_100_20250315.nc


<xarray.Dataset> Size: 63kB
Dimensions:     (valid_time: 24, latitude: 31, longitude: 21)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 192B 2025-03-15 ... 2025-03-15T23...
    expver      (valid_time) <U4 384B ...
  * latitude    (latitude) float64 248B 20.0 19.9 19.8 19.7 ... 17.2 17.1 17.0
  * longitude   (longitude) float64 168B 98.0 98.1 98.2 98.3 ... 99.8 99.9 100.0
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 62kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-01T07:45 GRIB to CDM+CF via cfgrib-0.9.1...

In [4]:
era5_utils.describe_era5(ds,name="2m_temperature — Chiang Mai, 15/03/2025")

OVERVIEW: 2m_temperature — Chiang Mai, 15/03/2025

[1] Dimensions: {'valid_time': 24, 'latitude': 31, 'longitude': 21}

[2] Coordinates:
    number: 0 
    valid_time: 24 values, from 2025-03-15T00:00:00.000000000 to 2025-03-15T23:00:00.000000000
    latitude: 31 values, from 20.0 to 17.0
    longitude: 21 values, from 98.0 to 100.0
    expver: 24 values, from 0001 to 0001

[3] Data variables (short name -> long name, units):
    t2m      -> 2 metre temperature [K]

[4] Value ranges:
    t2m      min=288.0229  max=312.1956



In [5]:
SAMPLE_LAT, SAMPLE_LON = 18.5, 99.0

point_df = era5_utils.extract_point(ds, latitude=SAMPLE_LAT, longitude=SAMPLE_LON)
point_df.head()

Requested (18.5, 99.0) -> matched grid cell (18.50, 99.00), ~0.0 km away


,time,t2m,latitude,longitude
0,2025-03-15 00:00:00,296.118652,18.5,99.0
1,2025-03-15 01:00:00,296.198975,18.5,99.0
2,2025-03-15 02:00:00,300.356567,18.5,99.0
3,2025-03-15 03:00:00,303.418823,18.5,99.0
4,2025-03-15 04:00:00,305.678467,18.5,99.0


In [10]:
ds_full = era5_api.download_era5(
    variables=list(config.ERA5_VARIABLES.keys()),
    area=config.BBOX_CHIANG_MAI_CDS,
    year=2025, month=3, day=15,
)

point_full = era5_utils.extract_point(ds_full, latitude=SAMPLE_LAT, longitude=SAMPLE_LON)
point_full = era5_utils.add_derived_variables(point_full)
point_full[["t2m", "temperature_c", "tp","swvl1", "u10", "v10", "wind_speed_ms"]].head()

[cache] Reading era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_temperature_total_precipitation_volumetric_soil_water_layer_1_20_98_17_100_20250315.nc
Requested (18.5, 99.0) -> matched grid cell (18.50, 99.00), ~0.0 km away


,t2m,temperature_c,tp,swvl1,u10,v10,wind_speed_ms
0,296.118652,22.968658,0.000002,0.146042,-0.165741,0.547714,0.572242
1,296.198975,23.048981,0.000000,0.146042,-0.163803,0.375702,0.409858
2,300.356567,27.206573,0.000000,0.146042,-0.055984,0.381567,0.385652
3,303.418823,30.268829,0.000000,0.145996,0.318253,0.398760,0.510190
4,305.678467,32.528473,0.000000,0.145935,0.756851,0.265846,0.802183


In [7]:
daily = era5_utils.daily_precipitation(point_full)
daily

,date,precipitation_mm
0,2025-03-15,0.001716


In [9]:
import xarray as xr
days = range(9, 16)                                 
datasets = [
    era5_api.download_era5(list(config.ERA5_VARIABLES.keys()),
                           config.BBOX_CHIANG_MAI_CDS, 2025, 3, d)
    for d in days
]
ds_week = xr.concat(datasets, dim="valid_time")       

point_week = era5_utils.extract_point(ds_week, 18.5, 99.0)
daily_week = era5_utils.daily_precipitation(point_week)    
rolled_week = era5_utils.rolling_precip_sum(daily_week, 7)
rolled_week

Submitting request to CDS - this can take from under a minute to a few hours depending on server load ...


601e186e6704822979f27a593ee786ef.nc:   0%|          | 0.00/195k [00:00<?, ?B/s]

[downloaded] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_temperature_total_precipitation_volumetric_soil_water_layer_1_20_98_17_100_20250309.nc
Submitting request to CDS - this can take from under a minute to a few hours depending on server load ...


839351a1274cde63f8bb9ebdab4539bf.nc:   0%|          | 0.00/194k [00:00<?, ?B/s]

[downloaded] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_temperature_total_precipitation_volumetric_soil_water_layer_1_20_98_17_100_20250310.nc
Submitting request to CDS - this can take from under a minute to a few hours depending on server load ...


...

5b2fceb6527cd7ce284e0424a6ddf13b.nc:   0%|          | 0.00/183k [00:00<?, ?B/s]

[downloaded] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_temperature_total_precipitation_volumetric_soil_water_layer_1_20_98_17_100_20250313.nc
Submitting request to CDS - this can take from under a minute to a few hours depending on server load ...


7ea38264f361d12babf2314116a7ceb5.nc:   0%|          | 0.00/181k [00:00<?, ?B/s]

[downloaded] era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_temperature_total_precipitation_volumetric_soil_water_layer_1_20_98_17_100_20250314.nc
[cache] Reading era5land_10m_u_component_of_wind_10m_v_component_of_wind_2m_temperature_total_precipitation_volumetric_soil_water_layer_1_20_98_17_100_20250315.nc
Requested (18.5, 99.0) -> matched grid cell (18.50, 99.00), ~0.0 km away


,date,precipitation_mm,precipitation_mm_7d
0,2025-03-09,0.107068,0.107068
1,2025-03-10,0.004567,0.111635
2,2025-03-11,0.000858,0.112493
3,2025-03-12,0.000858,0.113351
4,2025-03-13,0.000852,0.114204
5,2025-03-14,0.001723,0.115926
6,2025-03-15,0.001716,0.117643
